In [1]:
# ============================================================================
# 02 — BUILD THE BENCHMARK
# ----------------------------------------------------------------------------
# For each eligible protein we pick TWO partners:
#   - one where the protein does its USUAL thing   (habit agrees with truth)
#   - one where it does the OPPOSITE               (habit misleads)
# The two are matched on how well-documented they are, so any accuracy gap
# cannot be blamed on one being more obscure than the other.
# Output: data/processed/sighabit_v1.jsonl
# ============================================================================
import json, hashlib, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW, PROC = ROOT/"data"/"raw", ROOT/"data"/"processed"
PROC.mkdir(parents=True, exist_ok=True)

SEED       = 20260817
VERSION    = "v1"
MIN_REFS   = 2       # each edge needs >=2 supporting papers
MIN_SIGN   = 2       # protein needs >=2 targets of each sign
MIN_USUAL  = 0.70    # protein's default must be at least 70% one-directional
REF_TOL    = 0.60    # matched pair: log-reference counts within this distance
N_PER_GROUP = 100    # pairs per protein group

rng = np.random.RandomState(SEED)

# ---- 1. rebuild the edge table (same filters as notebook 01) -------------
def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s

NEEDS_DIR = {"causal_ppi": True, "kinaseextra": True, "collectri": False}
raw, parts = {}, []
for name in ["causal_ppi","collectri","kinaseextra"]:
    raw[name] = pd.read_parquet(RAW/f"{name}.parquet")
    d = raw[name].copy()
    if NEEDS_DIR[name]:
        d = d[d.is_directed.astype(bool)]
    stim, inh = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(stim ^ inh).values]
    d = d[~d.source_genesymbol.map(is_complex).values]
    d = d[~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values]
    d = d[~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"]   = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = d.n_references.fillna(0).astype(int) if "n_references" in d.columns else 0
    d["component"] = name
    parts.append(d[["source_genesymbol","target_genesymbol","source","target",
                    "sign","n_refs","component"]])

E = pd.concat(parts, ignore_index=True)
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
CONFLICTED = set(conf[conf>1].index)
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(CONFLICTED)]
E = E.sort_values("n_refs", ascending=False).drop_duplicates("_pk", keep="first")
print(f"usable signed edges: {len(E):,}  (removed {len(CONFLICTED)} sign-conflicted pairs)")

# ---- 2. per-protein habit, LEAVE-ONE-OUT ---------------------------------
# The habit must never include the edge we are about to ask about, or the
# answer leaks into the very variable we use to predict it.
tot = E.groupby("source_genesymbol").sign.agg(
    n="size", n_stim=lambda s: (s=="stimulates").sum())

def loo_share(gene, sign_of_this_edge):
    """This protein's activation rate over its OTHER partners."""
    n, k = tot.loc[gene, "n"], tot.loc[gene, "n_stim"]
    n -= 1
    if sign_of_this_edge == "stimulates": k -= 1
    return k/n if n > 0 else np.nan

# ---- 3. eligible proteins -------------------------------------------------
G = pd.read_csv(PROC/"eligible_proteins.csv", index_col=0)
G["lopsided"] = np.maximum(G.stim_share, 1-G.stim_share)
G["usual"]    = np.where(G.stim_share > .5, "stimulates", "inhibits")
G = G[G.lopsided >= MIN_USUAL]

GROUP = {"kinase":"kinase", "transcription factor":"tf",
         "phosphatase":"other_enzyme", "E3 ligase":"other_enzyme",
         "protease":"other_enzyme", "other":"other_enzyme"}
G["group"] = G.pclass.map(GROUP)
print(f"\neligible proteins at >={MIN_USUAL:.0%} usual: {len(G)}")
print("  " + G.group.value_counts().to_string().replace("\n","\n  "))

# ---- 4. build matched pairs ------------------------------------------------
# For every protein: take one USUAL-sign partner and one OPPOSITE-sign partner
# whose reference counts are as close as possible. Reject if we cannot match.
Eq = E[E.n_refs >= MIN_REFS]
pairs, rejected = [], {"no_match":0, "too_few":0}

for gene in rng.permutation(G.index):
    r = G.loc[gene]
    usual, opp = r.usual, ("inhibits" if r.usual=="stimulates" else "stimulates")
    U = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==usual)]
    O = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==opp)]
    if len(U) < 1 or len(O) < 1:
        rejected["too_few"] += 1; continue
    # match on log reference count
    best = None
    for _, o in O.iterrows():
        d = (np.log1p(U.n_refs) - np.log1p(o.n_refs)).abs()
        j = d.idxmin()
        if d[j] <= REF_TOL and (best is None or d[j] < best[0]):
            best = (d[j], U.loc[j], o)
    if best is None:
        rejected["no_match"] += 1; continue
    _, u_edge, o_edge = best
    pairs.append(dict(gene=gene, group=r.group, pclass=r.pclass,
                      usual=usual, lopsided=r.lopsided,
                      u_target=u_edge.target_genesymbol, u_refs=int(u_edge.n_refs),
                      u_comp=u_edge.component, u_acc=u_edge.target,
                      o_target=o_edge.target_genesymbol, o_refs=int(o_edge.n_refs),
                      o_comp=o_edge.component, o_acc=o_edge.target,
                      src_acc=u_edge.source))
P = pd.DataFrame(pairs)
print(f"\nmatched pairs built: {len(P)}   rejected: {rejected}")

# ---- 5. quota by group, and balance the direction of the habit -----------
keep = []
for grp, g in P.groupby("group"):
    n = min(N_PER_GROUP, len(g))
    # within a group, keep habit direction as balanced as availability allows
    inh = g[g.usual=="inhibits"]; sti = g[g.usual=="stimulates"]
    n_inh = min(len(inh), n//2)
    n_sti = n - n_inh
    sel = pd.concat([inh.iloc[rng.permutation(len(inh))[:n_inh]],
                     sti.iloc[rng.permutation(len(sti))[:n_sti]]])
    keep.append(sel)
    print(f"  {grp:<14}{len(sel):>4} pairs  "
          f"(usual=inhibits {n_inh}, usual=stimulates {n_sti}, available {len(g)})")
P = pd.concat(keep).reset_index(drop=True)
print(f"\nfinal pairs: {len(P)}   ->  {len(P)*2} questions")

# ---- 6. expand each pair into two items -----------------------------------
def iid(*a): return hashlib.sha1("|".join(map(str,a)).encode()).hexdigest()[:16]

items = []
for _, p in P.iterrows():
    for cond, tgt, refs, comp, acc, truth in [
        ("habit_agrees",   p.u_target, p.u_refs, p.u_comp, p.u_acc, p.usual),
        ("habit_misleads", p.o_target, p.o_refs, p.o_comp, p.o_acc,
         "inhibits" if p.usual=="stimulates" else "stimulates")]:
        items.append(dict(
            item_id=iid(p.gene, tgt), pair_id=iid(p.gene, "pair"),
            source=p.gene, target=tgt,
            source_uniprot=p.src_acc, target_uniprot=acc,
            answer=truth, condition=cond,
            protein_usually=p.usual, group=p.group, pclass=p.pclass,
            lopsided=round(float(p.lopsided), 3),
            n_refs=int(refs), component=comp,
            src_stim_share_loo=round(float(loo_share(p.gene, truth)), 4),
            is_decoy=False))
B = pd.DataFrame(items)

# ---- 7. verify the design --------------------------------------------------
print("\n" + "="*78); print("DESIGN CHECK"); print("="*78)
print(f"items {len(B)} | proteins {B.source.nunique()} | pairs {B.pair_id.nunique()}")

ct = pd.crosstab(B.protein_usually, B.answer)
print("\nthe four cells (rows = protein's habit, cols = true answer here):")
print(ct.to_string())
print(f"\noverall answer balance: "
      f"stimulates {(B.answer=='stimulates').mean():.1%} / "
      f"inhibits {(B.answer=='inhibits').mean():.1%}")

print("\nby protein group:")
print(pd.crosstab(B.group, [B.protein_usually, B.condition]).to_string())

print("\nreference counts — are the two conditions matched?")
for c in ["habit_agrees","habit_misleads"]:
    s = B[B.condition==c]
    print(f"  {c:<16} median {s.n_refs.median():>5.0f}   "
          f"mean {s.n_refs.mean():>6.1f}   range {s.n_refs.min()}-{s.n_refs.max()}")
from scipy.stats import mannwhitneyu
u = mannwhitneyu(B[B.condition=='habit_agrees'].n_refs,
                 B[B.condition=='habit_misleads'].n_refs)
print(f"  Mann-Whitney p = {u.pvalue:.3f}  (want >0.05: conditions well matched)")

print("\nsanity checks:")
chk = [
 ("every protein contributes exactly 2 items",
  (B.groupby('source').size()==2).all()),
 ("no duplicate item ids", B.item_id.is_unique),
 ("answers balanced within 2pp",
  abs((B.answer=='stimulates').mean()-.5) < .02),
 ("each pair has one agree + one mislead",
  (B.groupby('pair_id').condition.nunique()==2).all()),
 ("no sign-conflicted pairs included",
  not any((r.source,r.target) in CONFLICTED for _,r in B.iterrows())),
]
for name, ok in chk:
    print(f"  [{'OK ' if ok else 'BAD'}] {name}")
assert all(o for _,o in chk), "design check failed"

# ---- 8. write ---------------------------------------------------------------
path = PROC/f"sighabit_{VERSION}.jsonl"
with open(path, "w", encoding="utf-8") as f:
    for _, r in B.iterrows():
        f.write(json.dumps({k:(None if pd.isna(v) else
                (int(v) if isinstance(v,(np.integer,)) else
                 float(v) if isinstance(v,(np.floating,)) else
                 bool(v) if isinstance(v,(np.bool_,)) else v))
                for k,v in r.items()}) + "\n")
SHA = hashlib.sha256(path.read_bytes()).hexdigest()[:16]

(PROC/f"sighabit_{VERSION}.meta.json").write_text(json.dumps(dict(
    version=VERSION, seed=SEED, built_utc=datetime.datetime.now(
        datetime.timezone.utc).isoformat(), sha256=SHA,
    n_items=len(B), n_proteins=int(B.source.nunique()),
    min_refs=MIN_REFS, min_targets_per_sign=MIN_SIGN,
    min_usual_share=MIN_USUAL, ref_match_tolerance=REF_TOL,
    n_sign_conflicted_removed=len(CONFLICTED),
    components=["causal_ppi","collectri","kinaseextra"]), indent=2))

print(f"\nwritten: {path}")
print(f"sha256 : {SHA}")
print("\nexample items:")
for _, r in B.head(4).iterrows():
    print(f"  {r.source:>9} -> {r.target:<10} truth={r.answer:<11} "
          f"({r.condition}, usually {r.protein_usually}, {r.n_refs} refs)")

usable signed edges: 61,975  (removed 111 sign-conflicted pairs)

eligible proteins at >=70% usual: 201
  group
  tf              114
  kinase           52
  other_enzyme     35

matched pairs built: 201   rejected: {'no_match': 0, 'too_few': 0}
  kinase          52 pairs  (usual=inhibits 3, usual=stimulates 49, available 52)
  other_enzyme    35 pairs  (usual=inhibits 15, usual=stimulates 20, available 35)
  tf             100 pairs  (usual=inhibits 4, usual=stimulates 96, available 114)

final pairs: 187   ->  374 questions

DESIGN CHECK
items 374 | proteins 187 | pairs 187

the four cells (rows = protein's habit, cols = true answer here):
answer           inhibits  stimulates
protein_usually                      
inhibits               22          22
stimulates            165         165

overall answer balance: stimulates 50.0% / inhibits 50.0%

by protein group:
protein_usually     inhibits                  stimulates               
condition       habit_agrees habit_misleads habi

In [2]:
# ============================================================================
# 02b — REBALANCE: give the "usually inhibits" proteins enough weight
# ----------------------------------------------------------------------------
# v1 had 22 usually-inhibits vs 165 usually-stimulates proteins. The inhibit
# side is the control that rules out "models just over-say activates", so it
# needs to be big enough to measure. We lower the habit threshold for those
# proteins only, and cap how many stimulate-default proteins we take.
# ============================================================================
import json, hashlib, datetime
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import mannwhitneyu

ROOT = Path.cwd().parent
RAW, PROC = ROOT/"data"/"raw", ROOT/"data"/"processed"

SEED = 20260817
VERSION = "v2"
MIN_REFS      = 2
USUAL_STIM    = 0.70     # threshold for stimulate-default proteins
USUAL_INH     = 0.62     # relaxed for inhibit-default (they are rarer)
REF_TOL       = 0.60
MAX_STIM_MULT = 2.0      # stimulate-default proteins <= 2x the inhibit-default
rng = np.random.RandomState(SEED)

# ---- rebuild edges (identical filters to 02) ----------------------------
def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s
NEEDS_DIR = {"causal_ppi":True, "kinaseextra":True, "collectri":False}
parts = []
for name in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{name}.parquet")
    if NEEDS_DIR[name]: d = d[d.is_directed.astype(bool)]
    stim, inh = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(stim ^ inh).values]
    d = d[~d.source_genesymbol.map(is_complex).values]
    d = d[~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values]
    d = d[~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"]   = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = d.n_references.fillna(0).astype(int) if "n_references" in d.columns else 0
    d["component"] = name
    parts.append(d[["source_genesymbol","target_genesymbol","source","target",
                    "sign","n_refs","component"]])
E = pd.concat(parts, ignore_index=True)
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
CONFLICTED = set(conf[conf>1].index)
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(CONFLICTED)]
E = E.sort_values("n_refs", ascending=False).drop_duplicates("_pk", keep="first")

tot = E.groupby("source_genesymbol").sign.agg(
    n="size", n_stim=lambda s:(s=="stimulates").sum())
def loo_share(gene, sign_here):
    n, k = tot.loc[gene,"n"], tot.loc[gene,"n_stim"]
    n -= 1
    if sign_here == "stimulates": k -= 1
    return k/n if n > 0 else np.nan

# ---- eligible proteins, with the asymmetric threshold -------------------
G = pd.read_csv(PROC/"eligible_proteins.csv", index_col=0)
G["lopsided"] = np.maximum(G.stim_share, 1-G.stim_share)
G["usual"]    = np.where(G.stim_share > .5, "stimulates", "inhibits")
G = G[((G.usual=="stimulates") & (G.lopsided >= USUAL_STIM)) |
      ((G.usual=="inhibits")   & (G.lopsided >= USUAL_INH))]
GROUP = {"kinase":"kinase","transcription factor":"tf","phosphatase":"other_enzyme",
         "E3 ligase":"other_enzyme","protease":"other_enzyme","other":"other_enzyme"}
G["group"] = G.pclass.map(GROUP)
print("eligible proteins after asymmetric threshold:")
print(pd.crosstab(G.group, G.usual).to_string())

# ---- matched pairs -------------------------------------------------------
Eq = E[E.n_refs >= MIN_REFS]
pairs = []
for gene in rng.permutation(G.index):
    r = G.loc[gene]
    opp = "inhibits" if r.usual=="stimulates" else "stimulates"
    U = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==r.usual)]
    O = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==opp)]
    if not len(U) or not len(O): continue
    best = None
    for _, o in O.iterrows():
        dd = (np.log1p(U.n_refs) - np.log1p(o.n_refs)).abs()
        j = dd.idxmin()
        if dd[j] <= REF_TOL and (best is None or dd[j] < best[0]):
            best = (dd[j], U.loc[j], o)
    if best is None: continue
    _, u, o = best
    pairs.append(dict(gene=gene, group=r.group, pclass=r.pclass, usual=r.usual,
        lopsided=r.lopsided, u_target=u.target_genesymbol, u_refs=int(u.n_refs),
        u_comp=u.component, u_acc=u.target, o_target=o.target_genesymbol,
        o_refs=int(o.n_refs), o_comp=o.component, o_acc=o.target, src_acc=u.source))
P = pd.DataFrame(pairs)
print(f"\nmatched pairs available: {len(P)}")
print(pd.crosstab(P.group, P.usual).to_string())

# ---- CAP the stimulate-default side so the two are comparable ------------
n_inh = int((P.usual=="inhibits").sum())
n_sti_cap = int(n_inh * MAX_STIM_MULT)
keep = [P[P.usual=="inhibits"]]                       # take ALL inhibit-default
sti = P[P.usual=="stimulates"]
per_grp = max(1, n_sti_cap // sti.group.nunique())    # spread across groups
for grp, g in sti.groupby("group"):
    keep.append(g.iloc[rng.permutation(len(g))[:per_grp]])
P = pd.concat(keep).reset_index(drop=True)
print(f"\nafter capping: {len(P)} pairs  "
      f"(inhibits {int((P.usual=='inhibits').sum())}, "
      f"stimulates {int((P.usual=='stimulates').sum())})")

# ---- expand to items -----------------------------------------------------
def iid(*a): return hashlib.sha1("|".join(map(str,a)).encode()).hexdigest()[:16]
items = []
for _, p in P.iterrows():
    for cond, tgt, refs, comp, acc, truth in [
        ("habit_agrees",   p.u_target, p.u_refs, p.u_comp, p.u_acc, p.usual),
        ("habit_misleads", p.o_target, p.o_refs, p.o_comp, p.o_acc,
         "inhibits" if p.usual=="stimulates" else "stimulates")]:
        items.append(dict(item_id=iid(p.gene,tgt), pair_id=iid(p.gene,"pair"),
            source=p.gene, target=tgt, source_uniprot=p.src_acc, target_uniprot=acc,
            answer=truth, condition=cond, protein_usually=p.usual,
            group=p.group, pclass=p.pclass, lopsided=round(float(p.lopsided),3),
            n_refs=int(refs), component=comp,
            src_stim_share_loo=round(float(loo_share(p.gene,truth)),4), is_decoy=False))
B = pd.DataFrame(items)

# ---- verify ---------------------------------------------------------------
print("\n"+"="*78); print("DESIGN CHECK"); print("="*78)
print(f"items {len(B)} | proteins {B.source.nunique()} | pairs {B.pair_id.nunique()}")
print("\nthe four cells:")
print(pd.crosstab(B.protein_usually, B.answer).to_string())
print(f"\nanswer balance: stimulates {(B.answer=='stimulates').mean():.1%}")
print(f"habit direction: usually-inhibits {(B.protein_usually=='inhibits').mean():.1%} "
      f"(was 12% in v1)")
print("\nby group:")
print(pd.crosstab(B.group, [B.protein_usually, B.condition]).to_string())
print("\nreference matching:")
for c in ["habit_agrees","habit_misleads"]:
    s = B[B.condition==c]
    print(f"  {c:<16} median {s.n_refs.median():>4.0f}  mean {s.n_refs.mean():>5.1f}")
print(f"  Mann-Whitney p = "
      f"{mannwhitneyu(B[B.condition=='habit_agrees'].n_refs, B[B.condition=='habit_misleads'].n_refs).pvalue:.3f}")

print("\npower: smallest cell has "
      f"{pd.crosstab(B.protein_usually,B.answer).values.min()} questions")
print("  (~30 gives a +/-18pp interval; we expect a 40pp effect, so this is enough)")

chk = [("2 items per protein",(B.groupby('source').size()==2).all()),
       ("unique item ids", B.item_id.is_unique),
       ("answers balanced", abs((B.answer=='stimulates').mean()-.5) < .03),
       ("each pair agrees+misleads",(B.groupby('pair_id').condition.nunique()==2).all())]
for n,ok in chk: print(f"  [{'OK ' if ok else 'BAD'}] {n}")
assert all(o for _,o in chk)

path = PROC/f"sighabit_{VERSION}.jsonl"
with open(path,"w",encoding="utf-8") as f:
    for _, r in B.iterrows():
        f.write(json.dumps({k:(None if pd.isna(v) else
            int(v) if isinstance(v,np.integer) else
            float(v) if isinstance(v,np.floating) else
            bool(v) if isinstance(v,np.bool_) else v) for k,v in r.items()})+"\n")
SHA = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
(PROC/f"sighabit_{VERSION}.meta.json").write_text(json.dumps(dict(
    version=VERSION, seed=SEED, sha256=SHA, n_items=len(B),
    n_proteins=int(B.source.nunique()),
    built_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    min_refs=MIN_REFS, usual_threshold_stim=USUAL_STIM,
    usual_threshold_inh=USUAL_INH, ref_match_tolerance=REF_TOL,
    max_stim_multiple=MAX_STIM_MULT,
    note="inhibit-default proteins are rare, so their habit threshold is "
         "relaxed and stimulate-default proteins are capped, to keep both "
         "directions of the habit measurable"), indent=2))
print(f"\nwritten: {path}   sha={SHA}")
print("\nexamples from each cell:")
for u in ["stimulates","inhibits"]:
    for c in ["habit_agrees","habit_misleads"]:
        s = B[(B.protein_usually==u)&(B.condition==c)]
        if len(s):
            r = s.iloc[0]
            print(f"  usually {u:<11} {c:<15} {r.source:>9} -> {r.target:<9} "
                  f"= {r.answer} ({r.n_refs} refs)")

eligible proteins after asymmetric threshold:
usual         inhibits  stimulates
group                             
kinase               5          49
other_enzyme        24          20
tf                   9         110

matched pairs available: 217
usual         inhibits  stimulates
group                             
kinase               5          49
other_enzyme        24          20
tf                   9         110

after capping: 108 pairs  (inhibits 38, stimulates 70)

DESIGN CHECK
items 216 | proteins 108 | pairs 108

the four cells:
answer           inhibits  stimulates
protein_usually                      
inhibits               38          38
stimulates             70          70

answer balance: stimulates 50.0%
habit direction: usually-inhibits 35.2% (was 12% in v1)

by group:
protein_usually     inhibits                  stimulates               
condition       habit_agrees habit_misleads habit_agrees habit_misleads
group                                                

In [3]:
# ============================================================================
# 03 — ADD DECOYS AND THE "USUALLY" QUESTIONS
# ----------------------------------------------------------------------------
# Decoys: real proteins paired at random, screened against ALL of OmniPath so
#   we know they are not documented anywhere. Correct answer: not_established.
# "Usually" questions: one per protein, asking the model its own belief about
#   that protein's default behaviour. Asked in a SEPARATE session so it cannot
#   prime the specific questions.
# ============================================================================
import json, hashlib, datetime
from pathlib import Path
import pandas as pd, numpy as np
import omnipath as op

ROOT = Path.cwd().parent
RAW, PROC = ROOT/"data"/"raw", ROOT/"data"/"processed"
SEED, VERSION, N_DECOY = 20260817, "v2", 60
rng = np.random.RandomState(SEED + 1)

B = pd.read_json(PROC/f"sighabit_{VERSION}.jsonl", lines=True)
print(f"benchmark: {len(B)} items, {B.source.nunique()} proteins")

# ---- 1. everything OmniPath knows about, for screening ------------------
p_all = RAW/"all_interactions.parquet"
if p_all.exists():
    ALL = pd.read_parquet(p_all)
else:
    ALL = op.interactions.AllInteractions.get(organism="human", genesymbols=True)
    ALL.to_parquet(p_all, index=False)
print(f"full OmniPath table: {len(ALL):,} rows")

KNOWN = set()
for s, t in zip(ALL.source_genesymbol, ALL.target_genesymbol):
    KNOWN.add((s,t)); KNOWN.add((t,s))          # both directions
for name in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{name}.parquet")
    for s, t in zip(d.source_genesymbol, d.target_genesymbol):
        KNOWN.add((s,t)); KNOWN.add((t,s))
print(f"pairs known to OmniPath (either direction): {len(KNOWN):,}")

# ---- 2. build decoys from the SAME proteins -----------------------------
# Using our own proteins keeps fame and familiarity comparable to real items,
# which was a weakness of the previous study.
pool = sorted(set(B.source) | set(B.target))
acc  = {}
for _, r in B.iterrows():
    if isinstance(r.source_uniprot, str): acc.setdefault(r.source, r.source_uniprot)
    if isinstance(r.target_uniprot, str): acc.setdefault(r.target, r.target_uniprot)

def iid(*a): return hashlib.sha1("|".join(map(str,a)).encode()).hexdigest()[:16]

decoys, used, tries = [], set(), 0
while len(decoys) < N_DECOY and tries < 500_000:
    tries += 1
    i, j = rng.choice(len(pool), 2, replace=False)
    s, t = pool[i], pool[j]
    if (s,t) in KNOWN or (s,t) in used: continue
    used.add((s,t))
    decoys.append(dict(item_id=iid("decoy",s,t), pair_id=iid("decoy",s,t),
        source=s, target=t, source_uniprot=acc.get(s), target_uniprot=acc.get(t),
        answer="not_established", condition="decoy", protein_usually=None,
        group="decoy", pclass=None, lopsided=None, n_refs=0,
        component="decoy", src_stim_share_loo=None, is_decoy=True))
D = pd.DataFrame(decoys)
print(f"\ndecoys built: {len(D)} (after {tries:,} draws)")
assert not any((r.source,r.target) in KNOWN for _,r in D.iterrows()), "a decoy is real"

# how comparable are decoy proteins to real ones?
deg = pd.concat([pd.read_parquet(RAW/f"{n}.parquet") for n in
                 ["causal_ppi","collectri","kinaseextra"]]).groupby(
                 "source_genesymbol").size()
print("  source-protein connectivity (log): "
      f"real {np.log1p([deg.get(s,0) for s in B.source]).mean():.2f}  "
      f"decoy {np.log1p([deg.get(s,0) for s in D.source]).mean():.2f}")
print("  (close values mean refusal cannot be explained by obscurity alone)")

FULL = pd.concat([B, D], ignore_index=True)

# ---- 3. the "usually" questions -----------------------------------------
# One per protein. This elicits the model's OWN habit belief, so we can test
# whether the shortcut lives in the model or only in our database.
U = (B.drop_duplicates("source")[["source","source_uniprot","protein_usually",
                                  "group","pclass","lopsided"]]
       .rename(columns={"protein_usually":"answer"}).reset_index(drop=True))
U["item_id"]   = [iid("usually", g) for g in U.source]
U["condition"] = "usually"
U["target"]    = None
print(f"\n'usually' questions: {len(U)}  "
      f"({(U.answer=='inhibits').mean():.0%} of proteins usually inhibit)")

# ---- 4. write ------------------------------------------------------------
def clean(r):
    return {k:(None if (not isinstance(v,(list,dict)) and pd.isna(v)) else
        int(v) if isinstance(v,np.integer) else float(v) if isinstance(v,np.floating)
        else bool(v) if isinstance(v,np.bool_) else v) for k,v in r.items()}

for df, nm in [(FULL, f"sighabit_{VERSION}_full"), (U, f"sighabit_{VERSION}_usually")]:
    p = PROC/f"{nm}.jsonl"
    with open(p,"w",encoding="utf-8") as f:
        for _, r in df.iterrows(): f.write(json.dumps(clean(r))+"\n")
    print(f"  written {p.name}: {len(df)} items  "
          f"sha={hashlib.sha256(p.read_bytes()).hexdigest()[:16]}")

# ---- 5. the prompt -------------------------------------------------------
SYSTEM = ("You are an expert molecular biologist. Reply with a single JSON "
          "object and nothing else. No preamble, no markdown, no code fences.")
SCHEMA = ('{"answer": "stimulates" | "inhibits" | "not_established", '
          '"confidence": <integer 1-10>, "rationale": "<one sentence>"}')
SCHEMA_SWAP = ('{"answer": "inhibits" | "stimulates" | "not_established", '
               '"confidence": <integer 1-10>, "rationale": "<one sentence>"}')
GUIDE = ("Answer stimulates if the net effect is activating, inhibits if "
         "repressing, or not_established if no such direct relationship is "
         "documented.")

def render(item, swap=False):
    if item.get("condition") == "usually":
        body = (f"Considering all of its known direct targets, does {item['source']} "
                f"more often activate them or more often inhibit them?\n"
                f"Answer stimulates if it more often activates, inhibits if it "
                f"more often represses, or not_established if you do not know.")
    else:
        body = (f"What is the net regulatory effect of {item['source']} on "
                f"{item['target']} at the protein level?\n{GUIDE}")
    schema = SCHEMA_SWAP if swap else SCHEMA
    return f"{body}\n\nRespond with JSON only:\n{schema}"

(PROC/f"prompts_{VERSION}.json").write_text(json.dumps(dict(
    system=SYSTEM, schema=SCHEMA, schema_swapped=SCHEMA_SWAP, guide=GUIDE), indent=2))

print("\n" + "="*78); print("PROMPT EXAMPLES"); print("="*78)
for lab, it in [("habit agrees",  B[B.condition=="habit_agrees"].iloc[0].to_dict()),
                ("habit misleads",B[B.condition=="habit_misleads"].iloc[0].to_dict()),
                ("decoy",         D.iloc[0].to_dict()),
                ("usually",       U.iloc[0].to_dict())]:
    print(f"\n--- {lab} ---\n{render(it)}")

print("\n" + "="*78); print("FINAL BENCHMARK"); print("="*78)
print(f"  main questions   : {len(B)}  ({B.pair_id.nunique()} matched pairs)")
print(f"  fabricated pairs : {len(D)}")
print(f"  'usually' probes : {len(U)}  (asked separately)")
print(f"  TOTAL per model  : {len(FULL)} + {len(U)} = {len(FULL)+len(U)} calls")
print(f"\n  at ~160 in / ~250 out tokens per call:")
for m, pi, po in [("gemini-3.7-flash",.1875,.9375),("gpt-5.6-luna",.10,.60),
                  ("gpt-5.6-terra",1.00,6.00)]:
    c = (len(FULL)+len(U))*(160/1e6*pi + 250/1e6*po)
    print(f"    {m:<20} ${c:.3f}")

c:\Users\tauhe\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


benchmark: 216 items, 108 proteins
full OmniPath table: 290,512 rows
pairs known to OmniPath (either direction): 550,622

decoys built: 60 (after 63 draws)
  source-protein connectivity (log): real 4.28  decoy 3.13
  (close values mean refusal cannot be explained by obscurity alone)

'usually' questions: 108  (35% of proteins usually inhibit)
  written sighabit_v2_full.jsonl: 276 items  sha=af57414e8835ac87
  written sighabit_v2_usually.jsonl: 108 items  sha=941726bab5018025

PROMPT EXAMPLES

--- habit agrees ---
What is the net regulatory effect of NFIL3 on PTGS2 at the protein level?
Answer stimulates if the net effect is activating, inhibits if repressing, or not_established if no such direct relationship is documented.

Respond with JSON only:
{"answer": "stimulates" | "inhibits" | "not_established", "confidence": <integer 1-10>, "rationale": "<one sentence>"}

--- habit misleads ---
What is the net regulatory effect of NFIL3 on IL3 at the protein level?
Answer stimulates if the ne

In [4]:
# ============================================================================
# 03b — DECOYS x120 AND MATCHED WORDING
# ----------------------------------------------------------------------------
# Two fixes:
#  1. 120 decoys instead of 60, so refusal rates are measurable with precision
#  2. the "usually" question now uses the SAME phrasing as the main question
#     ("net regulatory effect", "stimulates"), so the two are directly
#     comparable and no wording difference can explain a difference in answers
# ============================================================================
import json, hashlib, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW, PROC = ROOT/"data"/"raw", ROOT/"data"/"processed"
SEED, VERSION, N_DECOY = 20260817, "v2", 120
rng = np.random.RandomState(SEED + 1)

B = pd.read_json(PROC/f"sighabit_{VERSION}.jsonl", lines=True)
ALL = pd.read_parquet(RAW/"all_interactions.parquet")
print(f"benchmark {len(B)} items, {B.source.nunique()} proteins | "
      f"OmniPath {len(ALL):,} rows")

KNOWN = set()
for s,t in zip(ALL.source_genesymbol, ALL.target_genesymbol):
    KNOWN.add((s,t)); KNOWN.add((t,s))
for n in ["causal_ppi","collectri","kinaseextra"]:
    d = pd.read_parquet(RAW/f"{n}.parquet")
    for s,t in zip(d.source_genesymbol, d.target_genesymbol):
        KNOWN.add((s,t)); KNOWN.add((t,s))
print(f"pairs known to OmniPath: {len(KNOWN):,}")

# ---- decoys, matched on connectivity -------------------------------------
# Previously decoys were less connected than real items, which meant refusal
# could reflect unfamiliarity rather than knowledge. Here we prefer decoy
# proteins whose connectivity resembles the real items'.
deg = pd.concat([pd.read_parquet(RAW/f"{n}.parquet")
                 for n in ["causal_ppi","collectri","kinaseextra"]]
                ).groupby("source_genesymbol").size()
pool = sorted(set(B.source) | set(B.target))
real_src_deg = np.log1p([deg.get(s,0) for s in B.source]).mean()

acc = {}
for _, r in B.iterrows():
    if isinstance(r.source_uniprot,str): acc.setdefault(r.source, r.source_uniprot)
    if isinstance(r.target_uniprot,str): acc.setdefault(r.target, r.target_uniprot)
def iid(*a): return hashlib.sha1("|".join(map(str,a)).encode()).hexdigest()[:16]

# oversample candidates, then keep the subset closest in connectivity
cand, used, tries = [], set(), 0
while len(cand) < N_DECOY*6 and tries < 2_000_000:
    tries += 1
    i,j = rng.choice(len(pool),2,replace=False); s,t = pool[i],pool[j]
    if (s,t) in KNOWN or (s,t) in used: continue
    used.add((s,t)); cand.append((s,t,np.log1p(deg.get(s,0))))
C = pd.DataFrame(cand, columns=["s","t","logdeg"])
C["gap"] = (C.logdeg - real_src_deg).abs()
C = C.sort_values("gap").head(N_DECOY)

D = pd.DataFrame([dict(item_id=iid("decoy",s,t), pair_id=iid("decoy",s,t),
    source=s, target=t, source_uniprot=acc.get(s), target_uniprot=acc.get(t),
    answer="not_established", condition="decoy", protein_usually=None,
    group="decoy", pclass=None, lopsided=None, n_refs=0, component="decoy",
    src_stim_share_loo=None, is_decoy=True) for s,t in zip(C.s, C.t)])

print(f"\ndecoys: {len(D)} (from {len(cand):,} candidates, {tries:,} draws)")
print(f"  source connectivity (log): real {real_src_deg:.2f}  "
      f"decoy {np.log1p([deg.get(s,0) for s in D.source]).mean():.2f}  "
      f"gap {np.log1p([deg.get(s,0) for s in D.source]).mean()-real_src_deg:+.2f}")
print("  (was -1.15 before matching; closer means refusal reflects knowledge,")
print("   not unfamiliarity)")
assert not any((r.source,r.target) in KNOWN for _,r in D.iterrows())
assert D.item_id.is_unique

FULL = pd.concat([B, D], ignore_index=True)

# ---- "usually" probes, now with MATCHED wording -------------------------
U = (B.drop_duplicates("source")[["source","source_uniprot","protein_usually",
                                  "group","pclass","lopsided"]]
       .rename(columns={"protein_usually":"answer"}).reset_index(drop=True))
U["item_id"], U["condition"], U["target"] = [iid("usually",g) for g in U.source], "usually", None
print(f"\n'usually' probes: {len(U)} ({(U.answer=='inhibits').mean():.0%} inhibit-default)")

# ---- prompts: identical vocabulary across both question types -----------
SYSTEM = ("You are an expert molecular biologist. Reply with a single JSON "
          "object and nothing else. No preamble, no markdown, no code fences.")
SCHEMA      = ('{"answer": "stimulates" | "inhibits" | "not_established", '
               '"confidence": <integer 1-10>, "rationale": "<one sentence>"}')
SCHEMA_SWAP = ('{"answer": "inhibits" | "stimulates" | "not_established", '
               '"confidence": <integer 1-10>, "rationale": "<one sentence>"}')
GUIDE = ("Answer stimulates if the net effect is activating, inhibits if "
         "repressing, or not_established if no such direct relationship is "
         "documented.")
GUIDE_USUAL = ("Answer stimulates if the net effect is more often activating, "
               "inhibits if more often repressing, or not_established if you "
               "do not know.")

def render(item, swap=False):
    if item.get("condition") == "usually":
        body = (f"Across its known direct targets, what is the net regulatory "
                f"effect of {item['source']} at the protein level?\n{GUIDE_USUAL}")
    else:
        body = (f"What is the net regulatory effect of {item['source']} on "
                f"{item['target']} at the protein level?\n{GUIDE}")
    return f"{body}\n\nRespond with JSON only:\n{SCHEMA_SWAP if swap else SCHEMA}"

(PROC/f"prompts_{VERSION}.json").write_text(json.dumps(dict(
    system=SYSTEM, schema=SCHEMA, schema_swapped=SCHEMA_SWAP,
    guide=GUIDE, guide_usually=GUIDE_USUAL,
    note="both question types use 'net regulatory effect ... at the protein "
         "level' and the same answer labels, so wording cannot explain a "
         "difference between them"), indent=2))

def clean(r):
    return {k:(None if (not isinstance(v,(list,dict)) and pd.isna(v)) else
        int(v) if isinstance(v,np.integer) else float(v) if isinstance(v,np.floating)
        else bool(v) if isinstance(v,np.bool_) else v) for k,v in r.items()}
for df, nm in [(FULL, f"sighabit_{VERSION}_full"), (U, f"sighabit_{VERSION}_usually")]:
    p = PROC/f"{nm}.jsonl"
    with open(p,"w",encoding="utf-8") as f:
        for _,r in df.iterrows(): f.write(json.dumps(clean(r))+"\n")
    print(f"  written {p.name}: {len(df)} items  "
          f"sha={hashlib.sha256(p.read_bytes()).hexdigest()[:16]}")

print("\n"+"="*78); print("PROMPTS — note the shared wording"); print("="*78)
for lab, it in [("main question", B.iloc[0].to_dict()),
                ("usually probe", U.iloc[0].to_dict()),
                ("decoy",         D.iloc[0].to_dict()),
                ("order swapped", B.iloc[0].to_dict())]:
    print(f"\n--- {lab} ---\n{render(it, swap=(lab=='order swapped'))}")

print("\n"+"="*78); print("FINAL"); print("="*78)
print(f"  main {len(B)} | decoys {len(D)} | usually {len(U)}  "
      f"-> {len(FULL)+len(U)} calls per model")
PLAN = [("gpt-5.6-terra",1.00,6.00),("gpt-5.6-luna",.10,.60),
        ("gemini-3.7-flash",.1875,.9375),("gpt-oss-120b",.03,.17),
        ("qwen3-32b",.10,.30)]
tot = 0
for m,pi,po in PLAN:
    c = (len(FULL)+len(U))*(160/1e6*pi + 250/1e6*po); tot += c
    print(f"    {m:<20} ${c:.3f}")
print(f"    {'order-swap (3 models)':<20} ${len(B)*3*(160/1e6*.4+250/1e6*2.5):.3f}")
print(f"    {'web search (Gemini)':<20} ${len(FULL)*4/1000 + len(FULL)*(160/1e6*.1875+250/1e6*.9375):.3f}")
print(f"\n  ESTIMATED TOTAL: ~${tot + 0.4 + 1.6:.2f}")

benchmark 216 items, 108 proteins | OmniPath 290,512 rows
pairs known to OmniPath: 550,622

decoys: 120 (from 720 candidates, 826 draws)
  source connectivity (log): real 4.28  decoy 4.29  gap +0.01
  (was -1.15 before matching; closer means refusal reflects knowledge,
   not unfamiliarity)

'usually' probes: 108 (35% inhibit-default)
  written sighabit_v2_full.jsonl: 336 items  sha=99be1e9522e4f933
  written sighabit_v2_usually.jsonl: 108 items  sha=941726bab5018025

PROMPTS — note the shared wording

--- main question ---
What is the net regulatory effect of NFIL3 on PTGS2 at the protein level?
Answer stimulates if the net effect is activating, inhibits if repressing, or not_established if no such direct relationship is documented.

Respond with JSON only:
{"answer": "stimulates" | "inhibits" | "not_established", "confidence": <integer 1-10>, "rationale": "<one sentence>"}

--- usually probe ---
Across its known direct targets, what is the net regulatory effect of NFIL3 at the protei